In [ ]:
!pip install datasets transformers huggingface_hub transformers[torch] accelerate --upgrade
import math


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from huggingface_hub import login

In [ ]:
login()

In [ ]:
import re
from sklearn.model_selection import train_test_split

In [ ]:
f = open("./input.txt", "r")
text = f.readlines()

In [ ]:
print(len(text))

19611


In [ ]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

print(train)
print(test)

Train dataset length: 16669
Test dataset length: 2942
['Drona married the sister of Kripa, and a\n', 'foot in the country. The rivers and the\n', 'return with the certainty that we shall\n', 'would give up your life when the kingdom\n', 'Krishna took a solemn vow before\n', 'giving them half the kingdom."\n', 'anywhere such vivid portraiture on so\n', 'teacher. Fool! When your hour comes,\n', 'Satyaki is indeed in the paws of the\n', 'burnt his body, mixed the ashes in wine\n', 'we do our duties energetically. Even a\n', '"Wretch!" replied Duryodhana. "Living, I\n', 'When they started talking about obtaining\n', 'Devayani and said: Dear daughter, here is\n', 'the Bharata race, you have done what no\n', 'When Dhananjaya left the main front for\n', "justice may be even, I ask that Madri's son\n", 'the Kaurava hospitality.\n', 'across this mighty river with your silly\n', 'committed in ignorance. If you can be my\n', "Bhishma personally opposed Arjuna's\n", 'to live incognito in Virata\'s

In [ ]:
tokeinzer = AutoTokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [ ]:
from transformers import TextDataset, DataCollatorForLanguageModeling

def load_dataset(train_path, test_path, tokenizer):
  train_dataset = TextDataset(tokenizer=tokenizer, file_path=train_path, block_size=64)

  test_dataset = TextDataset(tokenizer=tokenizer, file_path=test_path, block_size=64)

  data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokeinzer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (166212 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelWithLMHead

model = AutoModelWithLMHead.from_pretrained("gpt2")

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/modeling_auto.py:1748: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-mahabharata",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=64,
    save_steps=100,
    warmup_steps=50,
    per_device_eval_batch_size=64,
    evaluation_strategy="steps",
    save_total_limit=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=205, training_loss=4.028726121855945, metrics={'train_runtime': 277.982, 'train_samples_per_second': 46.712, 'train_steps_per_second': 0.737, 'total_flos': 424109629440000.0, 'train_loss': 4.028726121855945, 'epoch': 5.0})

In [ ]:
trainer.save_model()

In [ ]:
input_text = "Pandavas were   "
input_ids = tokeinzer.encode(input_text, return_tensors="pt").to("cuda")
output = model.generate(input_ids, max_length=100, num_return_sequences=1)

generated_text = tokeinzer.decode(output[0], skip_special_tokens=True)
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Karna was   the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the Pandavas.  the battle.  the battle.  the Pandavas.  the Pandavas.  the battle.  the Pandavas.  the Pandavas.


In [ ]:
model.save_pretrained("gpt2-mahabharata")
tokeinzer.save_pretrained("gpt2-mahabharata")

('gpt2-mahabharata/tokenizer_config.json',
 'gpt2-mahabharata/special_tokens_map.json',
 'gpt2-mahabharata/vocab.json',
 'gpt2-mahabharata/merges.txt',
 'gpt2-mahabharata/added_tokens.json',
 'gpt2-mahabharata/tokenizer.json')

In [ ]:
def evaluate_model(model, dataset):
  trainer = Trainer(
      model=model,
      args=training_args,
      data_collator=data_collator,
      eval_dataset=test_dataset,
  )
  eval_results = trainer.evaluate()
  return eval_results

eval_results_v1 = evaluate_model(model, test_dataset)

model_gpt2 = AutoModelWithLMHead.from_pretrained("gpt2")
eval_results_v2 = evaluate_model(model_gpt2, test_dataset)

print("Evaluation Results (v1):", eval_results_v1)
print("Evaluation Results (v2):", eval_results_v2)


/usr/local/lib/python3.10/dist-packages/transformers/models/auto/modeling_auto.py:1748: FutureWarning: The class `AutoModelWithLMHead` is deprecated and will be removed in a future version. Please use `AutoModelForCausalLM` for causal language models, `AutoModelForMaskedLM` for masked language models and `AutoModelForSeq2SeqLM` for encoder-decoder models.
  warnings.warn(


Evaluation Results (v1): {'eval_loss': 3.7642605304718018, 'eval_model_preparation_time': 0.0093, 'eval_runtime': 2.6763, 'eval_samples_per_second': 170.755, 'eval_steps_per_second': 2.989}
Evaluation Results (v2): {'eval_loss': 5.379091262817383, 'eval_model_preparation_time': 0.0044, 'eval_runtime': 2.6641, 'eval_samples_per_second': 171.537, 'eval_steps_per_second': 3.003}


In [ ]:
from huggingface_hub import login, create_repo, Repository
login()
repo_name = "fine-tuned-gpt2-mahabharata"  # Change this to your desired repository name
from huggingface_hub import HfApi

# Initialize the HfApi instance
api = HfApi(token="")

# Create a new repository
username = api.whoami()['name']  # Get your Hugging Face username
full_repo_name = f"{username}/{repo_name}"

# Create the repository (you can also create it on the Hugging Face website)
api.create_repo(repo_name, private=False)

api.upload_folder(
    folder_path='./gpt2-mahabharata',  # Path to the folder with your model
    repo_id=full_repo_name,  # Model repository name
    commit_message="GPT-2 Mahabharata"
)

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

Upload 11 LFS files:   0%|          | 0/11 [00:00<?, ?it/s]

optimizer.pt:   0%|          | 0.00/996M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

...

events.out.tfevents.1724420816.01995dfb5554.1481.1:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

events.out.tfevents.1724421415.01995dfb5554.1481.2:   0%|          | 0.00/360 [00:00<?, ?B/s]

events.out.tfevents.1724421419.01995dfb5554.1481.3:   0%|          | 0.00/360 [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/metriccoders/fine-tuned-gpt2-mahabharata/commit/5b2d25c633fb89c1d8741e74fa591afd2c692e46', commit_message='GPT-2 Mahabharata', commit_description='', oid='5b2d25c633fb89c1d8741e74fa591afd2c692e46', pr_url=None, pr_revision=None, pr_num=None)